Idea: Pseudo Online Evaluation of Deep Learning Models for Motor Imagery Direction Decoding Task
•	On Subject-1 Calibration data, split the data into train and test and compute the performance on Sub-1 Online Session data. 
•	For Sub-N, (N is between 2 to 20) 
o	Train data: Append all Sub-(N-1) Calibration data and correctly classified MI trials of Online session data
o	Test data: Sub-N Online Session Data

In [8]:
import scipy.io
import numpy as np
import os
import glob
import torch
from scipy import signal 
from sklearn.model_selection import LeaveOneOut
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

In [9]:
def load_mat_file(filepath):
    """ Load .mat file and return Xtr, Ytr always; Xte, Yte only if they are not NaN """
    mat_data = scipy.io.loadmat(filepath)
    
    Xtr = mat_data['Xtrain']
    Ytr = mat_data['Ytrain']
    Xte = mat_data['Xtest']
    Yte = mat_data['Ytest']
    Yte_fb = mat_data['Ytest_fb']
    
    # Check if Xte and Yte are NaN or entirely NaN arrays
    if np.isnan(Xte).all() or np.isnan(Yte).all() or np.isnan(Yte_fb).all():
        # If all values in Xte or Yte are NaN, return only Xtr and Ytr
        return Xtr, Ytr
    else:
        # Otherwise, return Xtr, Ytr, Xte, Yte
        return Xtr, Ytr, Xte, Yte, Yte_fb

# Baseline Correction 
def baseline_correction(X, baseline_samples=500):
    n_trails, n_samples, n_channels = X.shape
    Xbc = np.zeros_like(X)
    for t in range(n_trails):
        Xbase = X[t, :baseline_samples-1,:]
        Xeeg = X[t, baseline_samples:, :]
        Xbc[t, baseline_samples:, :] = Xeeg - np.mean(Xbase, axis=0) #baseline correction
    
    Xnew = Xbc[:, baseline_samples:, :]
    return Xnew

# Preprocessing: Surface Laplacian, Bandpass Filter
def bandpass_filtering(X, fs=500, fcut=[0.5, 20], filt_order=5):
    n_trials, n_samples, n_channels = X.shape
    X1 = np.zeros_like(X)
    b,a = signal.butter(filt_order, fcut, fs=fs, btype = 'band', output='ba') 
    for t in range(n_trials):
        for c in range(n_channels):
            #Error here.
            raw_signal = X[t, :, c]
            filt_signal = signal.filtfilt(b, a, raw_signal)
            X1[t, :, c] = filt_signal
            # Xfilt[t, :, c] = signal.filtfilt(b, a, X[t, :, c])
    return X1

In [10]:
def create_dataset(current_subject, base_path=None):
    """"
    Create training and test dataset for the current subjects.
    
    Parameters:
    current_subject (int): Subject number (1 to 20).
    base_path (str): The directory where .mat files are stored.

    Returns:
    X_train, Y_train, X_test, Y_test
    """
    Xtr_all = []
    Ytr_all = []
    for sub in range(1, current_subject+1):
        rel_path = f'data/S{sub:02d}_mitrials.mat'
        filepath = os.path.join(base_path, rel_path)

        mat_vars = load_mat_file(filepath)
        if len(mat_vars)==2:
            Xtr, Ytr = mat_vars
            Xtr_all.append(Xtr)
            Ytr_all.append(Ytr)

        else:
            Xtr, Ytr, Xte, Yte, Yte_fb = mat_vars
            Xtr_all.append(Xtr)
            Ytr_all.append(Ytr)
            if (sub == current_subject):
                continue
            else:
                Xtr_all.append(Xte)
                Ytr_all.append(Yte)
                # indx = np.where(Yte==Yte_fb)[0]
                # Xtr_all.append(Xte[indx, :, :])
                # Ytr_all.append(Yte[indx])
                
    X_train = np.concatenate(Xtr_all, axis=0)
    Y_train = np.concatenate(Ytr_all, axis=0)

    X_test = Xte
    Y_test = Yte
        
    return X_train, Y_train, X_test, Y_test



In [11]:
class EEGNet(nn.Module):
    def __init__(self, nb_classes, Chans=27, Samples=2500, dropoutRate=0.5, 
                 kernLength=64, F1=8, D=2, F2=16, norm_rate=0.25, dropoutType='Dropout'):
        super(EEGNet, self).__init__()
        
        # Handle dropout type
        if dropoutType == 'SpatialDropout2D':
            self.dropout = nn.Dropout2d(dropoutRate)
        elif dropoutType == 'Dropout':
            self.dropout = nn.Dropout(dropoutRate)
        else:
            raise ValueError('dropoutType must be one of SpatialDropout2D or Dropout.')

        # Block 1
        self.conv1 = nn.Conv2d(1, F1, (1, kernLength), padding='same', bias=False)
        self.batchnorm1 = nn.BatchNorm2d(F1)
        self.depthwiseConv = nn.Conv2d(F1, F1*D, (Chans, 1), groups=F1, bias=False)
        self.batchnorm2 = nn.BatchNorm2d(F1*D)
        self.pool1 = nn.AvgPool2d((1, 4))

        # Block 2
        self.separableConv = nn.Conv2d(F1*D, F2, (1, 16), padding='same', bias=False)
        self.batchnorm3 = nn.BatchNorm2d(F2)
        self.pool2 = nn.AvgPool2d((1, 8))

        # Flatten and Dense
        self.flatten = nn.Flatten()
        self.dense = nn.Linear(F2 * (Samples // (4 * 8)), nb_classes)
        self.norm_constraint = nn.utils.weight_norm(self.dense)

    def forward(self, x):
        # Block 1
        x = self.conv1(x)
        x = self.batchnorm1(x)
        x = self.depthwiseConv(x)
        x = self.batchnorm2(x)
        x = F.elu(x)
        x = self.pool1(x)
        x = self.dropout(x)

        # Block 2
        x = self.separableConv(x)
        x = self.batchnorm3(x)
        x = F.elu(x)
        x = self.pool2(x)
        x = self.dropout(x)

        # Flatten and Dense
        x = self.flatten(x)
        x = self.dense(x)
        return F.softmax(x, dim=1)

In [12]:
# Function to compute accuracy
def compute_accuracy(model, data_loader, device):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == targets.squeeze()).sum().item()
            total += targets.size(0)
    
    accuracy = correct / total
    return accuracy


In [13]:
torch.manual_seed(0)
parent_dir = os.path.dirname(os.getcwd())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

batch_size = 32
num_epochs = 500
fs = 500

perf = dict()
for sub in range(8, 21):
    # print(f'Current Subject : S{sub:02d}')
    Xtr, Ytr, Xte, Yte = create_dataset(sub, base_path=parent_dir)
    
    X_train = baseline_correction(Xtr)
    X_train = bandpass_filtering(X_train)
    X_test = baseline_correction(Xte)
    X_test = bandpass_filtering(X_test)

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1).permute(0, 1, 3, 2).to(device)  # Shape: (batch_size, 1, 27, 2500)
    Y_train_tensor = torch.tensor(Ytr, dtype=torch.long).to(device)  # Use long for classification
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1).permute(0, 1, 3, 2).to(device)  # Shape: (batch_size, 1, 27, 2500)
    Y_test_tensor = torch.tensor(Yte, dtype=torch.long).to(device)

    train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)


    train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

    # Example usage
    nb_classes = 2
    Chans = 27
    Samples = 2000
    dropoutRate = 0.5
    kernLength = 64
    F1 = 8
    D = 2
    F2 = 16
    norm_rate = 0.25
    dropoutType = 'Dropout'

    model = EEGNet(nb_classes=2, Chans=27, Samples=2000).to(device)

    criterion = nn.CrossEntropyLoss().to(device)  # Move loss function to GPU if needed
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    # Training loop
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            # outputs = model(inputs.permute(0, 2, 1))  # Permute to match Conv1D input shape
            loss = criterion(outputs, targets.squeeze())
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
    
    # Compute accuracy on the test dataset
    accuracy = compute_accuracy(model, test_loader, device)
    perf[f'Sub{sub}'] = accuracy
    
    print(f'Test Subject: S{sub}, Test Accuracy: {accuracy:.4f}')

print(perf)

Test Subject: S8, Test Accuracy: 0.5000
Test Subject: S9, Test Accuracy: 0.4375
Test Subject: S10, Test Accuracy: 0.5208
Test Subject: S11, Test Accuracy: 0.6250
Test Subject: S12, Test Accuracy: 0.4792
Test Subject: S13, Test Accuracy: 0.5417
Test Subject: S14, Test Accuracy: 0.5417
Test Subject: S15, Test Accuracy: 0.3958
Test Subject: S16, Test Accuracy: 0.5000
Test Subject: S17, Test Accuracy: 0.5000
Test Subject: S18, Test Accuracy: 0.5833
Test Subject: S19, Test Accuracy: 0.5000
Test Subject: S20, Test Accuracy: 0.6875
{'Sub8': 0.5, 'Sub9': 0.4375, 'Sub10': 0.5208333333333334, 'Sub11': 0.625, 'Sub12': 0.4791666666666667, 'Sub13': 0.5416666666666666, 'Sub14': 0.5416666666666666, 'Sub15': 0.3958333333333333, 'Sub16': 0.5, 'Sub17': 0.5, 'Sub18': 0.5833333333333334, 'Sub19': 0.5, 'Sub20': 0.6875}


In [14]:
mean_acc = list(perf.values())
print(mean_acc)
print(f'Average Accuracy: {np.mean(mean_acc)}')

[0.5, 0.4375, 0.5208333333333334, 0.625, 0.4791666666666667, 0.5416666666666666, 0.5416666666666666, 0.3958333333333333, 0.5, 0.5, 0.5833333333333334, 0.5, 0.6875]
Average Accuracy: 0.5240384615384616
